In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hmmlearn import hmm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Regime Detection Module Initialized")

## 1. Generate Market Data with Regime Changes

In [ ]:
# Generate synthetic data with distinct market regimes
np.random.seed(42)
n_days = 2000
dates = pd.date_range(start='2018-01-01', periods=n_days, freq='D')

# Define regime parameters
regimes = {
    'bull': {'mean': 0.001, 'vol': 0.008, 'duration': 400},
    'bear': {'mean': -0.0008, 'vol': 0.015, 'duration': 300},
    'sideways': {'mean': 0.0001, 'vol': 0.006, 'duration': 350},
    'crisis': {'mean': -0.002, 'vol': 0.03, 'duration': 150}
}

# Generate returns with regime switches
returns = []
true_regimes = []
regime_sequence = ['bull', 'sideways', 'bull', 'crisis', 'bear', 'sideways', 'bull']

for regime_name in regime_sequence:
    regime = regimes[regime_name]
    n = min(regime['duration'], n_days - len(returns))
    
    regime_returns = np.random.normal(regime['mean'], regime['vol'], n)
    returns.extend(regime_returns)
    true_regimes.extend([regime_name] * n)
    
    if len(returns) >= n_days:
        break

returns = np.array(returns[:n_days])
true_regimes = true_regimes[:n_days]

# Create price series
prices = 100 * np.exp(np.cumsum(returns))

# Create DataFrame
market_data = pd.DataFrame({
    'date': dates,
    'price': prices,
    'returns': returns,
    'true_regime': true_regimes
})
market_data.set_index('date', inplace=True)

# Calculate features for regime detection
market_data['volatility'] = market_data['returns'].rolling(window=20).std()
market_data['trend'] = market_data['returns'].rolling(window=50).mean()
market_data['momentum'] = market_data['price'].pct_change(periods=20)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price with regime coloring
regime_colors = {'bull': 'green', 'bear': 'red', 'sideways': 'gray', 'crisis': 'darkred'}
for regime in regime_colors.keys():
    regime_mask = market_data['true_regime'] == regime
    axes[0].scatter(market_data.index[regime_mask], market_data['price'][regime_mask], 
                   c=regime_colors[regime], s=5, alpha=0.6, label=regime.capitalize())
axes[0].plot(market_data.index, market_data['price'], 'k-', alpha=0.2, linewidth=0.5)
axes[0].set_ylabel('Price')
axes[0].set_title('Market Price with True Regimes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Returns
axes[1].plot(market_data.index, market_data['returns'], 'b-', alpha=0.5, linewidth=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Daily Returns')
axes[1].set_title('Daily Returns')
axes[1].grid(True, alpha=0.3)

# Rolling volatility
axes[2].plot(market_data.index, market_data['volatility'] * 100, 'orange', linewidth=1.5)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Volatility (%)')
axes[2].set_title('20-Day Rolling Volatility')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Generated {n_days} days of market data")
print(f"\nRegime distribution:")
print(market_data['true_regime'].value_counts())
print(f"\nData range: {market_data.index[0]} to {market_data.index[-1]}")
print(f"Price range: ${market_data['price'].min():.2f} to ${market_data['price'].max():.2f}")

## 2. Hidden Markov Model (HMM) Regime Detection

In [ ]:
# Prepare data for HMM (use returns)
returns_clean = market_data['returns'].dropna().values.reshape(-1, 1)

# Fit Gaussian HMM with 4 hidden states
n_states = 4
model_hmm = hmm.GaussianHMM(n_components=n_states, covariance_type="full", 
                            n_iter=1000, random_state=42)
model_hmm.fit(returns_clean)

# Predict hidden states
hidden_states = model_hmm.predict(returns_clean)

# Add to DataFrame (accounting for NaN from rolling calculations)
market_data_clean = market_data.dropna()
market_data_clean['hmm_regime'] = hidden_states[:len(market_data_clean)]

# Analyze regime characteristics
regime_stats = market_data_clean.groupby('hmm_regime').agg({
    'returns': ['mean', 'std', 'count'],
    'volatility': 'mean'
}).round(6)

# Sort regimes by mean return
regime_order = market_data_clean.groupby('hmm_regime')['returns'].mean().sort_values(ascending=False).index
regime_labels = {regime_order[0]: 'Bull', regime_order[1]: 'Neutral', 
                regime_order[2]: 'Bear', regime_order[3]: 'Crisis'}
market_data_clean['hmm_regime_label'] = market_data_clean['hmm_regime'].map(regime_labels)

# Get transition matrix
transition_matrix = model_hmm.transmat_

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Price with HMM regimes
hmm_colors = {0: 'green', 1: 'blue', 2: 'orange', 3: 'red'}
for state in range(n_states):
    mask = market_data_clean['hmm_regime'] == state
    axes[0, 0].scatter(market_data_clean.index[mask], market_data_clean['price'][mask],
                      c=hmm_colors[state], s=10, alpha=0.6, 
                      label=f'State {state} ({regime_labels[state]})')
axes[0, 0].plot(market_data_clean.index, market_data_clean['price'], 'k-', alpha=0.2, linewidth=0.5)
axes[0, 0].set_ylabel('Price')
axes[0, 0].set_title('HMM Detected Regimes')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Return distributions by regime
for state in range(n_states):
    state_returns = market_data_clean[market_data_clean['hmm_regime'] == state]['returns']
    axes[0, 1].hist(state_returns, bins=30, alpha=0.5, label=regime_labels[state], density=True)
axes[0, 1].set_xlabel('Returns')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Return Distributions by HMM Regime')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Regime statistics
regime_means = [market_data_clean[market_data_clean['hmm_regime'] == s]['returns'].mean() * 100 
               for s in range(n_states)]
regime_vols = [market_data_clean[market_data_clean['hmm_regime'] == s]['returns'].std() * 100 
              for s in range(n_states)]

x = np.arange(n_states)
width = 0.35
axes[1, 0].bar(x - width/2, regime_means, width, label='Mean Return', alpha=0.8, edgecolor='black')
axes[1, 0].bar(x + width/2, regime_vols, width, label='Volatility', alpha=0.8, edgecolor='black')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].set_title('Regime Characteristics')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([regime_labels[s] for s in range(n_states)])
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Transition probability matrix heatmap
transition_df = pd.DataFrame(transition_matrix, 
                            index=[regime_labels[i] for i in range(n_states)],
                            columns=[regime_labels[i] for i in range(n_states)])
sns.heatmap(transition_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1, 1],
           cbar_kws={'label': 'Probability'}, square=True)
axes[1, 1].set_title('Regime Transition Probability Matrix')
axes[1, 1].set_xlabel('To Regime')
axes[1, 1].set_ylabel('From Regime')

# Regime duration analysis
regime_changes = (market_data_clean['hmm_regime'].diff() != 0).cumsum()
regime_durations = market_data_clean.groupby([regime_changes, 'hmm_regime']).size()

duration_by_regime = {}
for (_, regime), duration in regime_durations.items():
    if regime_labels[regime] not in duration_by_regime:
        duration_by_regime[regime_labels[regime]] = []
    duration_by_regime[regime_labels[regime]].append(duration)

axes[2, 0].boxplot([duration_by_regime[label] for label in ['Bull', 'Neutral', 'Bear', 'Crisis']],
                   labels=['Bull', 'Neutral', 'Bear', 'Crisis'])
axes[2, 0].set_ylabel('Duration (days)')
axes[2, 0].set_title('Regime Duration Distribution')
axes[2, 0].grid(True, alpha=0.3, axis='y')

# Regime probability over time
state_probs = model_hmm.predict_proba(returns_clean[:len(market_data_clean)])
for state in range(n_states):
    axes[2, 1].plot(market_data_clean.index, state_probs[:, state], 
                   linewidth=1.5, label=regime_labels[state], alpha=0.7)
axes[2, 1].set_xlabel('Date')
axes[2, 1].set_ylabel('Probability')
axes[2, 1].set_title('Regime Probabilities Over Time')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Hidden Markov Model Results ===")
print(f"\nNumber of states: {n_states}")
print(f"\nRegime statistics:")
print(regime_stats)
print(f"\nTransition probability matrix:")
print(transition_df.round(3))
print(f"\nAverage regime durations:")
for label, durations in duration_by_regime.items():
    print(f"  {label}: {np.mean(durations):.1f} days (median: {np.median(durations):.1f})")

## 3. K-Means Clustering

In [ ]:
# Prepare features for clustering
features = market_data[['returns', 'volatility', 'trend', 'momentum']].dropna()

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Apply K-Means clustering
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(features_scaled)

# Add clusters to DataFrame
features['cluster'] = clusters

# Analyze cluster characteristics
cluster_stats = features.groupby('cluster').agg({
    'returns': ['mean', 'std'],
    'volatility': 'mean',
    'trend': 'mean',
    'momentum': 'mean'
}).round(6)

# Label clusters based on characteristics
cluster_order = features.groupby('cluster')['returns'].mean().sort_values(ascending=False).index
cluster_labels = {cluster_order[0]: 'High Growth', cluster_order[1]: 'Stable', 
                 cluster_order[2]: 'Volatile', cluster_order[3]: 'Decline'}
features['cluster_label'] = features['cluster'].map(cluster_labels)

# PCA for visualization
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features_scaled)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PCA scatter plot
scatter_colors = ['green', 'blue', 'orange', 'red']
for i, cluster in enumerate(range(n_clusters)):
    mask = clusters == cluster
    axes[0, 0].scatter(features_pca[mask, 0], features_pca[mask, 1], 
                      c=scatter_colors[i], label=cluster_labels[cluster], 
                      alpha=0.6, s=30, edgecolors='black', linewidths=0.5)

# Plot cluster centers
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0, 0].scatter(centers_pca[:, 0], centers_pca[:, 1], 
                  c='black', marker='X', s=300, edgecolors='white', linewidths=2, 
                  label='Centroids', zorder=5)
axes[0, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0, 0].set_title('K-Means Clusters (PCA Projection)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Price with cluster coloring
price_with_clusters = market_data['price'].loc[features.index]
for i, cluster in enumerate(range(n_clusters)):
    mask = features['cluster'] == cluster
    axes[0, 1].scatter(features.index[mask], price_with_clusters[mask],
                      c=scatter_colors[i], s=10, alpha=0.6, label=cluster_labels[cluster])
axes[0, 1].plot(features.index, price_with_clusters, 'k-', alpha=0.2, linewidth=0.5)
axes[0, 1].set_ylabel('Price')
axes[0, 1].set_title('Price with K-Means Clusters')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Cluster characteristics radar chart
cluster_means = features.groupby('cluster')[['returns', 'volatility', 'trend', 'momentum']].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

categories = ['Returns', 'Volatility', 'Trend', 'Momentum']
for cluster in range(n_clusters):
    values = cluster_means_norm.loc[cluster].values.tolist()
    values += values[:1]  # Complete the circle
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    
    axes[1, 0] = plt.subplot(2, 2, 3, projection='polar')
    axes[1, 0].plot(angles, values, 'o-', linewidth=2, label=cluster_labels[cluster])
    axes[1, 0].fill(angles, values, alpha=0.15)

axes[1, 0].set_xticks(angles[:-1])
axes[1, 0].set_xticklabels(categories)
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_title('Cluster Characteristics (Normalized)', y=1.08)
axes[1, 0].legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
axes[1, 0].grid(True)

# Feature importance for clustering
# Calculate variance of cluster centers for each feature
feature_importance = np.var(kmeans.cluster_centers_, axis=0)
feature_names = ['Returns', 'Volatility', 'Trend', 'Momentum']

axes[1, 1].barh(feature_names, feature_importance, alpha=0.8, color='purple', edgecolor='black')
axes[1, 1].set_xlabel('Variance of Cluster Centers')
axes[1, 1].set_title('Feature Importance for Clustering')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n=== K-Means Clustering Results ===")
print(f"\nNumber of clusters: {n_clusters}")
print(f"\nCluster statistics:")
print(cluster_stats)
print(f"\nCluster distribution:")
print(features['cluster_label'].value_counts())
print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}% (first 2 components)")

## 4. Volatility Regime Detection

In [ ]:
# Calculate different volatility measures
market_data['realized_vol'] = market_data['returns'].rolling(window=20).std() * np.sqrt(252)
market_data['parkinson_vol'] = np.sqrt(
    market_data['returns'].rolling(window=20).apply(
        lambda x: np.sum(x**2) / len(x) * 252
    )
)

# Define volatility regimes using percentiles
vol_data = market_data['realized_vol'].dropna()
low_vol_threshold = vol_data.quantile(0.33)
high_vol_threshold = vol_data.quantile(0.67)

def classify_vol_regime(vol):
    if pd.isna(vol):
        return np.nan
    elif vol < low_vol_threshold:
        return 'Low Volatility'
    elif vol > high_vol_threshold:
        return 'High Volatility'
    else:
        return 'Medium Volatility'

market_data['vol_regime'] = market_data['realized_vol'].apply(classify_vol_regime)

# Calculate regime-conditional returns
regime_performance = market_data.groupby('vol_regime')['returns'].agg([
    'mean', 'std', 'count',
    lambda x: (x > 0).sum() / len(x)  # Win rate
]).round(6)
regime_performance.columns = ['Mean Return', 'Std Dev', 'Count', 'Win Rate']

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Price with volatility regimes
vol_colors = {'Low Volatility': 'green', 'Medium Volatility': 'blue', 'High Volatility': 'red'}
for regime, color in vol_colors.items():
    mask = market_data['vol_regime'] == regime
    axes[0, 0].scatter(market_data.index[mask], market_data['price'][mask],
                      c=color, s=10, alpha=0.6, label=regime)
axes[0, 0].plot(market_data.index, market_data['price'], 'k-', alpha=0.2, linewidth=0.5)
axes[0, 0].set_ylabel('Price')
axes[0, 0].set_title('Price with Volatility Regimes')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Realized volatility over time
axes[0, 1].plot(market_data.index, market_data['realized_vol'] * 100, 'purple', linewidth=1)
axes[0, 1].axhline(y=low_vol_threshold * 100, color='green', linestyle='--', 
                  linewidth=2, label=f'Low: {low_vol_threshold*100:.1f}%')
axes[0, 1].axhline(y=high_vol_threshold * 100, color='red', linestyle='--', 
                  linewidth=2, label=f'High: {high_vol_threshold*100:.1f}%')
axes[0, 1].set_ylabel('Annualized Volatility (%)')
axes[0, 1].set_title('Realized Volatility with Regime Thresholds')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Returns by volatility regime
for regime in ['Low Volatility', 'Medium Volatility', 'High Volatility']:
    regime_returns = market_data[market_data['vol_regime'] == regime]['returns']
    axes[1, 0].hist(regime_returns, bins=30, alpha=0.5, label=regime, density=True)
axes[1, 0].set_xlabel('Returns')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Return Distributions by Volatility Regime')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Regime performance metrics
metrics = ['Mean Return', 'Std Dev', 'Win Rate']
x = np.arange(len(regime_performance.index))
width = 0.25

for i, metric in enumerate(metrics):
    values = regime_performance[metric].values
    if metric == 'Mean Return':
        values = values * 252 * 100  # Annualized %
    elif metric == 'Std Dev':
        values = values * np.sqrt(252) * 100  # Annualized %
    else:
        values = values * 100  # Percentage
    
    axes[1, 1].bar(x + i*width, values, width, label=metric, alpha=0.8, edgecolor='black')

axes[1, 1].set_ylabel('Value')
axes[1, 1].set_title('Performance Metrics by Volatility Regime')
axes[1, 1].set_xticks(x + width)
axes[1, 1].set_xticklabels(['Low Vol', 'Med Vol', 'High Vol'])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Volatility persistence
vol_changes = market_data['vol_regime'].dropna()
transition_counts = pd.crosstab(vol_changes, vol_changes.shift(-1), normalize='index')

sns.heatmap(transition_counts, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[2, 0],
           cbar_kws={'label': 'Probability'}, square=True)
axes[2, 0].set_title('Volatility Regime Transition Matrix')
axes[2, 0].set_xlabel('To Regime')
axes[2, 0].set_ylabel('From Regime')

# Cumulative returns by regime
for regime in ['Low Volatility', 'Medium Volatility', 'High Volatility']:
    regime_data = market_data[market_data['vol_regime'] == regime]
    cumulative = (1 + regime_data['returns']).cumprod()
    axes[2, 1].plot(regime_data.index, cumulative, linewidth=2, label=regime, alpha=0.7)

axes[2, 1].set_xlabel('Date')
axes[2, 1].set_ylabel('Cumulative Return')
axes[2, 1].set_title('Cumulative Returns by Volatility Regime')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Volatility Regime Analysis ===")
print(f"\nVolatility thresholds:")
print(f"  Low: < {low_vol_threshold*100:.2f}%")
print(f"  Medium: {low_vol_threshold*100:.2f}% - {high_vol_threshold*100:.2f}%")
print(f"  High: > {high_vol_threshold*100:.2f}%")
print(f"\nRegime performance (annualized):")
print(regime_performance)
print(f"\nRegime distribution:")
print(market_data['vol_regime'].value_counts())

## 5. Regime-Conditional Strategy Performance

In [ ]:
# Define simple strategies
def momentum_strategy(data, lookback=50):
    """Go long if price > MA, else flat"""
    ma = data['price'].rolling(window=lookback).mean()
    signal = (data['price'] > ma).astype(int)
    strategy_returns = signal.shift(1) * data['returns']
    return strategy_returns

def mean_reversion_strategy(data, lookback=20, num_std=2):
    """Buy when price < lower band, sell when price > upper band"""
    ma = data['price'].rolling(window=lookback).mean()
    std = data['price'].rolling(window=lookback).std()
    upper = ma + num_std * std
    lower = ma - num_std * std
    
    signal = pd.Series(0, index=data.index)
    signal[data['price'] < lower] = 1  # Buy oversold
    signal[data['price'] > upper] = -1  # Sell overbought
    signal = signal.fillna(method='ffill').fillna(0)
    
    strategy_returns = signal.shift(1) * data['returns']
    return strategy_returns

# Calculate strategy returns
market_data['momentum_returns'] = momentum_strategy(market_data)
market_data['mean_reversion_returns'] = mean_reversion_strategy(market_data)

# Analyze performance by HMM regime
market_data_with_hmm = market_data.join(market_data_clean[['hmm_regime_label']], how='inner')

regime_strategy_performance = market_data_with_hmm.groupby('hmm_regime_label').agg({
    'returns': lambda x: (x.mean() * 252 * 100, x.std() * np.sqrt(252) * 100),
    'momentum_returns': lambda x: (x.mean() * 252 * 100, x.std() * np.sqrt(252) * 100),
    'mean_reversion_returns': lambda x: (x.mean() * 252 * 100, x.std() * np.sqrt(252) * 100)
})

# Extract means and stds
performance_summary = []
for regime in ['Bull', 'Neutral', 'Bear', 'Crisis']:
    if regime in regime_strategy_performance.index:
        row_data = regime_strategy_performance.loc[regime]
        performance_summary.append({
            'Regime': regime,
            'Buy & Hold Return': row_data['returns'][0],
            'Buy & Hold Vol': row_data['returns'][1],
            'Momentum Return': row_data['momentum_returns'][0],
            'Momentum Vol': row_data['momentum_returns'][1],
            'Mean Reversion Return': row_data['mean_reversion_returns'][0],
            'Mean Reversion Vol': row_data['mean_reversion_returns'][1]
        })

performance_df = pd.DataFrame(performance_summary).set_index('Regime')

# Calculate Sharpe ratios
for strategy in ['Buy & Hold', 'Momentum', 'Mean Reversion']:
    performance_df[f'{strategy} Sharpe'] = (
        performance_df[f'{strategy} Return'] / performance_df[f'{strategy} Vol']
    )

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Strategy returns by regime
regimes = performance_df.index
x = np.arange(len(regimes))
width = 0.25

axes[0, 0].bar(x - width, performance_df['Buy & Hold Return'], width, 
              label='Buy & Hold', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x, performance_df['Momentum Return'], width, 
              label='Momentum', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x + width, performance_df['Mean Reversion Return'], width, 
              label='Mean Reversion', alpha=0.8, edgecolor='black')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].set_ylabel('Annualized Return (%)')
axes[0, 0].set_title('Strategy Returns by Regime')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(regimes)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Sharpe ratios by regime
axes[0, 1].bar(x - width, performance_df['Buy & Hold Sharpe'], width, 
              label='Buy & Hold', alpha=0.8, edgecolor='black')
axes[0, 1].bar(x, performance_df['Momentum Sharpe'], width, 
              label='Momentum', alpha=0.8, edgecolor='black')
axes[0, 1].bar(x + width, performance_df['Mean Reversion Sharpe'], width, 
              label='Mean Reversion', alpha=0.8, edgecolor='black')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].set_ylabel('Sharpe Ratio')
axes[0, 1].set_title('Sharpe Ratios by Regime')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(regimes)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Cumulative returns
cum_bh = (1 + market_data['returns']).cumprod()
cum_momentum = (1 + market_data['momentum_returns'].fillna(0)).cumprod()
cum_mr = (1 + market_data['mean_reversion_returns'].fillna(0)).cumprod()

axes[1, 0].plot(market_data.index, cum_bh, linewidth=2, label='Buy & Hold')
axes[1, 0].plot(market_data.index, cum_momentum, linewidth=2, label='Momentum')
axes[1, 0].plot(market_data.index, cum_mr, linewidth=2, label='Mean Reversion')
axes[1, 0].set_ylabel('Cumulative Return')
axes[1, 0].set_title('Strategy Cumulative Returns')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Rolling Sharpe ratios
window = 252
rolling_sharpe_momentum = (
    market_data['momentum_returns'].rolling(window=window).mean() /
    market_data['momentum_returns'].rolling(window=window).std() *
    np.sqrt(252)
)
rolling_sharpe_mr = (
    market_data['mean_reversion_returns'].rolling(window=window).mean() /
    market_data['mean_reversion_returns'].rolling(window=window).std() *
    np.sqrt(252)
)

axes[1, 1].plot(market_data.index, rolling_sharpe_momentum, linewidth=2, label='Momentum')
axes[1, 1].plot(market_data.index, rolling_sharpe_mr, linewidth=2, label='Mean Reversion')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Rolling Sharpe Ratio')
axes[1, 1].set_title(f'{window}-Day Rolling Sharpe Ratio')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Regime-Conditional Strategy Performance ===")
print("\nAnnualized returns and volatility by regime:")
print(performance_df.round(2))

print("\nOverall strategy performance:")
for strategy, returns_col in [('Buy & Hold', 'returns'), 
                             ('Momentum', 'momentum_returns'),
                             ('Mean Reversion', 'mean_reversion_returns')]:
    strat_returns = market_data[returns_col].dropna()
    total_return = (1 + strat_returns).prod() - 1
    ann_return = strat_returns.mean() * 252
    ann_vol = strat_returns.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    print(f"\n{strategy}:")
    print(f"  Total return: {total_return*100:.2f}%")
    print(f"  Annualized return: {ann_return*100:.2f}%")
    print(f"  Annualized volatility: {ann_vol*100:.2f}%")
    print(f"  Sharpe ratio: {sharpe:.2f}")

## 6. Summary

### Key Techniques:

1. **Hidden Markov Models (HMM)**
   - Probabilistic regime detection
   - Transition probability matrices
   - Regime probability estimation

2. **K-Means Clustering**
   - Feature-based regime classification
   - Unsupervised learning
   - Multi-dimensional analysis

3. **Volatility Regimes**
   - Threshold-based classification
   - Regime persistence analysis
   - Conditional performance metrics

4. **Regime-Conditional Strategies**
   - Strategy performance by regime
   - Adaptive strategy selection
   - Risk-adjusted returns

### Applications:
- Strategy adaptation based on market conditions
- Risk management (position sizing by regime)
- Portfolio allocation adjustments
- Early warning systems
- Performance attribution

### Best Practices:
- Use multiple regime detection methods
- Validate regimes with out-of-sample data
- Consider regime transition probabilities
- Adapt strategies to detected regimes
- Monitor regime stability over time
- Combine with other signals for robustness